# Fine-tuning LLM: Analýza českých písňových textů

Cílem tohoto projektu je vytvořit dataset a následně provést fine-tune jazykového modelu, který dokáže komplexně analyzovat texty českých písní.

Připravili sme si malý dataset 200 reálných textů písní od 4 specifických interpretů (Karel Kryl, Karel Plíhal, Ivan Mládek, Svěrák & Uhlíř). U každé skladby v datasetu máme její název, interpreta a hrubý text, kde občas po hrubém scrapingu zůstali akordy, nastavení kapodastru atp.

Využijeme ChatGPT API k vyčištění i sjednocené dat a vytvoření "inteligentních labelů" . Pro každý stažený text necháme velký model vygenerovat:
* **Stručné shrnutí** obsahu jednou větou.
* **Analýzu nálady** na škále 1 (nejvíce smutná) až 5 (nejvíce veselá).
* **Alternativní název**

Při této analýze bereme v potaz i samotného autora, protože každý z nich má jinou úroveň vážnosti či sarkasmu. Výsledný dataset bude sloužit k natrénování modelu, který se naučí tyto tři atributy ( shrnutí, nálada, laternativní název) generovat současně pouze na základě vstupního textu.

In [1]:


# 1. KONFIGURACE
API_KEY = "sk..."   # <--- VLOŽ SVŮJ OPENAI API KLÍČ
# URL na tvůj RAW soubor z GitHubu
GITHUB_RAW_URL = "https://raw.githubusercontent.com/LuciaKajanova/dspracticum25_flowers_team/refs/heads/main/homework/data_pisni_fin.jsonl"
TEMP_FILE = "rozpracovano_faze2_cista_metadata.jsonl"

# 2. INSTALACE A IMPORTY
print("Instaluji knihovny...")
!pip install -q openai tqdm requests

import requests
import json
import time
import os
import random
import re
from openai import OpenAI
from tqdm.notebook import tqdm

# 3. STAŽENÍ DAT Z GITHUBU
print(f"Stahuji data z GitHubu: {GITHUB_RAW_URL}")
raw_dataset = []
try:
    response = requests.get(GITHUB_RAW_URL)
    if response.status_code == 200:
        for line in response.text.strip().split('\n'):
            if line.strip():
                try: raw_dataset.append(json.loads(line))
                except: continue
        print(f"Úspěšně načteno {len(raw_dataset)} položek.")
    else:
        print(f"CHYBA stahování: Kód {response.status_code}. Zkontrolujte RAW URL.")
except Exception as e:
    print(f"Kritická chyba: {e}")

# 4. HLAVNÍ SMYČKA (API)
if raw_dataset and API_KEY.startswith("sk-"):
    client = OpenAI(api_key=API_KEY)
    final_data = []

    # Načtení rozpracované práce
    if os.path.exists(TEMP_FILE):
        with open(TEMP_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                try: final_data.append(json.loads(line))
                except: continue
        print(f"Navazuji na {len(final_data)} hotových.")

    with open(TEMP_FILE, 'a', encoding='utf-8') as f_out:
        for i, entry in enumerate(tqdm(raw_dataset, desc="AI Analýza")):
            if i < len(final_data): continue

            # Hrubé předčištění textu (pro lepší výsledek od GPT)
            txt = entry.get('text', '')
            txt = re.sub(r'\|\(.+?\)|\[.+?\]', '', txt)

            # --- API PROMPT (s opravenými uvozovkami) ---
            prompt = f"""
           Jsi hudební redaktor. Tvým úkolem je vyčistit text písně a vytvořit nová metadata.

            Vstupní text je plný smetí. Musíš odstranit:
            - VŠECHNY akordy (G, Emi7, Hmi/5- atd.)
            - Hudební poznámky (capo, int:, Rec:, Ref:, 2x)
            - Divné znaky (/, -, [ ])

            Vstup:
            Interpret: {entry.get('interpret')}
            Název: {entry.get('nazev')}
            Špinavý text: "{txt[:1500]}"

            Vytvoř JSON:
            1. "cisty_text": Pouze čistá slova písně.
            2. "alt_nazev": Vymysli krátký alternativní název.
            3. "shrnuti": Jedna česká věta o obsahu.
            4. "nalada": Číslo 1 (smutná) až 5 (veselá).

            Odpověz POUZE čistým JSON objektem.
            """

            try:
                resp = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.3, max_tokens=600
                )
                meta = json.loads(resp.choices[0].message.content.strip().replace("```json", "").replace("```", ""))

                # FINÁLNÍ FORMÁT PRO TRÉNINK (Výstup jen nová metadata)
                final_entry = {
                    "instruction": "Přečti si kontext, vyčisti text a vytvoř alternativní název, shrnutí (1 větu) a ohodnoť náladu (1-5).",
                    "input": f"Interpret: {entry.get('interpret')}\nNázev: {entry.get('nazev')}\nText:\n{meta['cisty_text']}",
                    "output": f"Alt. název: {meta['alt_nazev']} | Shrnutí: {meta['shrnuti']} | Nálada: {meta['nalada']}/5"
                }

                json.dump(final_entry, f_out, ensure_ascii=False)
                f_out.write('\n')
                f_out.flush()
                final_dataset.append(final_entry)
                time.sleep(0.05)
            except: continue

    # 5. ULOŽENÍ
    if len(final_dataset) >= 50:
        print("\nRozděluji data (train/test)...")
        random.shuffle(final_dataset)

        # Natvrdo 100ks train a 100ks test (pokud je dost dat)
        train_count = min(100, len(final_dataset))
        test_count = min(100, len(final_dataset) - train_count)

        train = final_dataset[:train_count]
        test = final_dataset[train_count:train_count + test_count]

        with open("train.jsonl", "w", encoding="utf-8") as f:
            for e in train: json.dump(e, f, ensure_ascii=False); f.write('\n')
        with open("test.jsonl", "w", encoding="utf-8") as f:
            for e in test: json.dump(e, f, ensure_ascii=False); f.write('\n')

        print(f"HOTOVO. train.jsonl ({len(train)} ks), test.jsonl ({len(test)} ks) připraveny.")
    else:
        print(f"Zatím málo dat ({len(final_dataset)} ks) na rozdělení.")

else:
    print("CHYBA: Chybí API klíč nebo vstupní data.")

Instaluji knihovny...
Stahuji data z GitHubu: https://raw.githubusercontent.com/LuciaKajanova/dspracticum25_flowers_team/refs/heads/main/homework/data_pisni_fin.jsonl
Úspěšně načteno 200 položek.


AI Analýza:   0%|          | 0/200 [00:00<?, ?it/s]

NameError: name 'final_dataset' is not defined

níže je skript na fix (problém se vyskytl kvůli nekontistenci názvů souborů v kodu výše) zachránění alespoň 199 textů. Použilo se načtení dat s temp_file (raději už znovu nepřeprocáváme skript a nepouziváme api klíč abychom neplýtvali limitem)

In [2]:
import json
import random
import os

TEMP_FILE = "rozpracovano_faze2_cista_metadata.jsonl"
print(f"Načítám finální data z: {TEMP_FILE}")

# Zde budeme mít výsledný seznam pro rozdělení
data_k_rozdeleni = []

if os.path.exists(TEMP_FILE):
    with open(TEMP_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data_k_rozdeleni.append(json.loads(line))
            except:
                continue
    print(f"Úspěšně načteno {len(data_k_rozdeleni)} hotových příkladů.")
else:
    print("CHYBA: Dočasný soubor nebyl nalezen. Zkus zkontrolovat Colab složku.")

Načítám finální data z: rozpracovano_faze2_cista_metadata.jsonl
Úspěšně načteno 199 hotových příkladů.


In [4]:
import json
import random
import os

# --- KONFIGURACE ---
TEMP_FILE = "rozpracovano_faze2_cista_metadata.jsonl"
TRAIN_COUNT = 100
TEST_COUNT = 99
OUTPUT_TRAIN = "train_100.jsonl"
OUTPUT_TEST = "test_99.jsonl"

# --- 1. NAČTENÍ DAT ---
print(f"Načítám finální data z: {TEMP_FILE}")

data_k_rozdeleni = []

if os.path.exists(TEMP_FILE):
    with open(TEMP_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data_k_rozdeleni.append(json.loads(line))
            except:
                continue
    print(f"Úspěšně načteno {len(data_k_rozdeleni)} hotových příkladů.")
else:
    print("CHYBA: Dočasný soubor nebyl nalezen. Ujisti se, že je ve složce Colabu.")

# --- 2. ROZDĚLENÍ A ULOŽENÍ ---
if len(data_k_rozdeleni) >= (TRAIN_COUNT + TEST_COUNT):
    print(f"\nProvádím finální rozdělení: {TRAIN_COUNT} trénink / {TEST_COUNT} test...")

    # Zamíchání dat pro náhodné rozdělení
    random.shuffle(data_k_rozdeleni)

    # Přesný řez
    train_data = data_k_rozdeleni[:TRAIN_COUNT]
    test_data = data_k_rozdeleni[TRAIN_COUNT:TRAIN_COUNT + TEST_COUNT]

    # Uložení trénovací sady
    with open(OUTPUT_TRAIN, "w", encoding="utf-8") as f:
        for e in train_data:
            json.dump(e, f, ensure_ascii=False)
            f.write('\n')

    # Uložení testovací sady
    with open(OUTPUT_TEST, "w", encoding="utf-8") as f:
        for e in test_data:
            json.dump(e, f, ensure_ascii=False)
            f.write('\n')

    print("--- VÝSLEDEK ---")
    print(f" HOTOVO! Soubor {OUTPUT_TRAIN} ({len(train_data)} ks) připraven.")
    print(f" HOTOVO! Soubor {OUTPUT_TEST} ({len(test_data)} ks) připraven.")
    print("Nyní můžeš tyto soubory použít v UnSloth notebooku (Fáze 3).")

else:
    print(f"\nCHYBA: Máš jen {len(data_k_rozdeleni)} příkladů. Není dostatek pro rozdělení 100/99.")

Načítám finální data z: rozpracovano_faze2_cista_metadata.jsonl
Úspěšně načteno 199 hotových příkladů.

Provádím finální rozdělení: 100 trénink / 99 test...
--- VÝSLEDEK ---
✅ HOTOVO! Soubor train_100.jsonl (100 ks) připraven.
✅ HOTOVO! Soubor test_99.jsonl (99 ks) připraven.
Nyní můžeš tyto soubory použít v UnSloth notebooku (Fáze 3).
